In [1]:
11

11

In [2]:
%load_ext autoreload
%autoreload 2

import sentiments_utils as utils
import torch
import os
from datasets import load_dataset, DatasetDict, Dataset, concatenate_datasets, disable_progress_bar
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    set_seed,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score, confusion_matrix
import numpy as np

2025-11-18 14:36:02.467612: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-18 14:36:02.518774: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-18 14:36:03.638274: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
print(f"📊 Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

📊 Device: GPU
   GPU: NVIDIA GeForce RTX 4070 Ti SUPER
   Memory: 17.17 GB


In [4]:
# Dataset configuration
DATASET_PATH = "projected_datasets"
DATASET_SPLITS = {"train": f"{DATASET_PATH}/train-robertuito.parquet",
                  "test_validation": f"{DATASET_PATH}/validation-reviewed.parquet"}
# Language configuration
SOURCE_COLUMN = "shp" 
TARGET_COLUMN = "spa"
NUMBER_LABELS = 3  # Positive, Negative, Neutral
# General configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training configuration
CONFIDENCE_THRESHOLD = 0.6
MAX_LENGTH = 128
SEED = 42
set_seed(SEED)

# 1. Preparar Datos

In [5]:
# Cargar datasets
full_dataset = utils.load_full_dataset(DATASET_PATH, DATASET_SPLITS)
train_dataset = full_dataset["train"]
test_validation_dataset = full_dataset["test_validation"]

# Limpiar datasets
train_dataset = utils.dedupe_bilingual(train_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))
test_validation_dataset = utils.dedupe_bilingual(test_validation_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))
# Remover overlaps entre train y validation
train_dataset = utils.remove_overlaps_bilingual(train_dataset, test_validation_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))

# Separar test y validation usando 50% para cada uno
test_validation_dataset = test_validation_dataset.train_test_split(test_size=0.5, seed=42)
test_dataset = test_validation_dataset["test"]
validation_dataset = test_validation_dataset["train"]

train_dataset, test_dataset, validation_dataset


📥 Loading dataset: projected_datasets

📊 Dataset Statistics:
   train: 16505 examples
   Example Shipibo: Jato shinamawe mesko yokabo axon neskaakin....
   Example Spanish: Ahora hazles recordar a través de diferentes pregu...
   test_validation: 1924 examples
   Example Shipibo: Metsara iwanke....
   Example Spanish: Fue maravilloso....


Filter:   0%|          | 0/10080 [00:00<?, ? examples/s]

(Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 8613
 }),
 Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 921
 }),
 Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 920
 }))

# 2. Preparar modelos

### Modelo 1: mmBERT-base


In [18]:
# Modelo base
base_model_name_1 =  "jhu-clsp/mmBERT-base"
base_tokenizer_name_1 =  "jhu-clsp/mmBERT-base"
base_model_path_1 = f"models/{base_model_name_1}"
base_tokenizer_path_1 = f"tokenizers/{base_model_name_1}"

# Modelo generado
trained_model_name_1 =  "mmBERT-sentiment-shp"
trained_model_path_1 = f"models/{trained_model_name_1}"
trained_tokenizer_path_1 = f"tokenizers/{trained_model_name_1}"

In [7]:
# Download Base model and tokenizer if not already present
if not os.path.exists(base_model_path_1):
    utils.download_model(base_model_name_1, base_model_path_1)
base_model_1 = utils.prepare_model(base_model_path_1, NUMBER_LABELS)
if not os.path.exists(base_tokenizer_path_1):
    utils.download_tokenizer(base_tokenizer_name_1, base_tokenizer_path_1)
base_tokenizer_1 = utils.prepare_tokenizer(base_tokenizer_path_1)
print("Model num_labels:", base_model_1.config.num_labels)


⬇️  Downloading model: jhu-clsp/mmBERT-base


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model saved to: models/jhu-clsp/mmBERT-base

⬇️  Loading model: models/jhu-clsp/mmBERT-base


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at models/jhu-clsp/mmBERT-base and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model loaded from: models/jhu-clsp/mmBERT-base

🔧 Loading tokenizer: tokenizers/jhu-clsp/mmBERT-base
   Tokenizer loaded from: tokenizers/jhu-clsp/mmBERT-base
Model num_labels: 3


### Modelo 2: xlm-roBERTa-base

In [17]:
# Modelo base
base_model_name_2 =  "FacebookAI/xlm-roberta-base"
base_tokenizer_name_2 =  "FacebookAI/xlm-roberta-base"
base_model_path_2 = f"models/{base_model_name_2}"
base_tokenizer_path_2 = f"tokenizers/{base_model_name_2}"

# Modelo generado
trained_model_name_2 =  "xlm-roberta-sentiment-shp"
trained_model_path_2 = f"models/{trained_model_name_2}"
trained_tokenizer_path_2 = f"tokenizers/{trained_model_name_2}"

In [9]:
# Download Base model and tokenizer if not already present
if not os.path.exists(base_model_path_2):
    utils.download_model(base_model_name_2, base_model_path_2)
base_model_2 = utils.prepare_model(base_model_path_2, NUMBER_LABELS)
if not os.path.exists(base_tokenizer_path_2):
    utils.download_tokenizer(base_tokenizer_name_2, base_tokenizer_path_2)
base_tokenizer_2 = utils.prepare_tokenizer(base_tokenizer_path_2)
print("Model num_labels:", base_model_2.config.num_labels)


⬇️  Downloading model: FacebookAI/xlm-roberta-base


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at models/FacebookAI/xlm-roberta-base and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Model saved to: models/FacebookAI/xlm-roberta-base

⬇️  Loading model: models/FacebookAI/xlm-roberta-base
   Model loaded from: models/FacebookAI/xlm-roberta-base

🔧 Loading tokenizer: tokenizers/FacebookAI/xlm-roberta-base
   Tokenizer loaded from: tokenizers/FacebookAI/xlm-roberta-base
Model num_labels: 3


# 3. Entrenar modelos

In [10]:
# Training hyperparameters
TRAIN_BATCH_SIZE = 4           # Adjust based on GPU memory
EVAL_BATCH_SIZE = 4          # Adjust based on GPU memory
LEARNING_RATE = 1e-5
NUM_EPOCHS = 10
WARMUP_STEPS = 500
SEED = 42

In [11]:
# Training arguments
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3, 
    early_stopping_threshold=0.001,
)

training_args = TrainingArguments(
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=WARMUP_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=50,
    report_to="wandb",  # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available,
    remove_unused_columns=False,  # Important for custom collator
)

callbacks=[early_stopping_callback]

In [12]:
def train_model(base_model, base_tokenizer, train_dataset, val_dataset, test_dataset, output_dir, training_args : TrainingArguments, callbacks):
    """Train the sentiment analysis model."""
    
    
    utils.clear_gpu_memory()

    
    def preprocess_function(dataset: Dataset):
        """Tokenize and encode the Shipibo texts."""
        encodings = base_tokenizer(dataset["shp"], truncation=True, padding=False, max_length=256)
        # Map sentiment labels to IDs
        label_map = {'NEG': 0, 'NEU': 1, 'POS': 2}
        encodings['label'] = [label_map[label] for label in dataset['label']]
        return encodings

    # Metrics function
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        accuracy = accuracy_score(labels, predictions)
        precision = precision_score(labels, predictions, average='weighted', zero_division=0)
        recall = recall_score(labels, predictions, average='weighted', zero_division=0)
        f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
        balanced_acc = balanced_accuracy_score(labels, predictions)
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'balanced_accuracy': balanced_acc
        }
    
    def prepare_dataset(dataset: Dataset):
        """Prepare dataset for training/evaluation."""
        dataset = dataset.map(preprocess_function, batched=True)
        columns_to_remove = [ x for x in dataset.column_names if x not in ['input_ids', 'attention_mask', 'label']]
        dataset = dataset.remove_columns(columns_to_remove)
        print("Dataset labels:", set(dataset["label"]))
        return dataset

    
    # Tokenize datasets
    tokenized_train = prepare_dataset(train_dataset)
    tokenized_validation = prepare_dataset(val_dataset)
    tokenized_test = prepare_dataset(test_dataset)
      
    # Initialize Trainer
    training_args.output_dir = output_dir
    trainer = Trainer(
        model=base_model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_validation,
        tokenizer=base_tokenizer,
        compute_metrics=compute_metrics,
        callbacks=callbacks,
    )

    # Train the model
    trainer.train()

    # Evaluate on test set
    test_results = trainer.evaluate(eval_dataset=tokenized_test)
    print("Test Results:", test_results)

    # Save the trained model and tokenizer
    trainer.save_model(output_dir)
    base_tokenizer.save_pretrained(output_dir)
    
    utils.clear_gpu_memory()
    return trainer, test_results

### Modelo 1

In [15]:
trainer_1, test_results_1 = train_model(
    base_model=base_model_1,
    base_tokenizer=base_tokenizer_1,
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    test_dataset=test_dataset,
    output_dir=trained_model_path_1,
    training_args=training_args,
    callbacks=callbacks
)

🧹 GPU memory cleared


Map:   0%|          | 0/8613 [00:00<?, ? examples/s]

Dataset labels: {0, 1, 2}


Map:   0%|          | 0/920 [00:00<?, ? examples/s]

Dataset labels: {0, 1, 2}


...

Dataset labels: {0, 1, 2}


/tmp/ipykernel_5351/636535234.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: Currently logged in as: davidoviedop (davidoviedop-aaa) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W1118 14:36:31.489000 5351 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Balanced Accuracy
1,0.594100,0.631770,0.752174,0.750992,0.752174,0.743750,0.616761
2,0.750000,0.749014,0.745652,0.776523,0.745652,0.756408,0.709130
3,0.528600,1.486221,0.768478,0.773892,0.768478,0.770060,0.694418
4,0.135100,2.502924,0.780435,0.779532,0.780435,0.779761,0.693743
5,0.070800,2.964512,0.760870,0.783883,0.760870,0.767972,0.715621
6,0.218000,2.799631,0.783696,0.777993,0.783696,0.779584,0.679904
7,0.000200,2.808043,0.789130,0.785162,0.789130,0.786756,0.693886
8,0.002100,2.922269,0.794565,0.792547,0.794565,0.792492,0.703402


Test Results: {'eval_loss': 3.112133026123047, 'eval_accuracy': 0.7350705754614549, 'eval_precision': 0.7492701904266704, 'eval_recall': 0.7350705754614549, 'eval_f1': 0.7396726394961425, 'eval_balanced_accuracy': 0.6840734683650256, 'eval_runtime': 6.9426, 'eval_samples_per_second': 132.659, 'eval_steps_per_second': 33.273, 'epoch': 8.0}
🧹 GPU memory cleared


In [16]:
trainer_2, test_results_2 = train_model(
    base_model=base_model_2,
    base_tokenizer=base_tokenizer_2,
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    test_dataset=test_dataset,
    output_dir=trained_model_path_2,
    training_args=training_args,
    callbacks=callbacks
)

🧹 GPU memory cleared


Map:   0%|          | 0/8613 [00:00<?, ? examples/s]

Dataset labels: {0, 1, 2}


Map:   0%|          | 0/920 [00:00<?, ? examples/s]

Dataset labels: {0, 1, 2}


Map:   0%|          | 0/921 [00:00<?, ? examples/s]

Dataset labels: {0, 1, 2}


/tmp/ipykernel_5351/636535234.py:49: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Balanced Accuracy
1,0.740600,0.671093,0.726087,0.708708,0.726087,0.709757,0.559616
2,0.748800,0.613166,0.738043,0.739288,0.738043,0.736069,0.638071
3,0.621200,0.724781,0.757609,0.778403,0.757609,0.752136,0.643902
4,0.641800,0.705888,0.783696,0.787445,0.783696,0.782549,0.694942
5,0.539000,0.816731,0.789130,0.784931,0.789130,0.786522,0.691326
6,0.493600,0.866367,0.792391,0.789865,0.792391,0.790917,0.698925
7,0.561900,1.049649,0.781522,0.787054,0.781522,0.783678,0.704656
8,0.475100,1.084625,0.794565,0.799774,0.794565,0.795852,0.729389
9,0.411800,1.149557,0.801087,0.801383,0.801087,0.800907,0.722523
10,0.663400,1.227148,0.792391,0.797039,0.792391,0.793300,0.718484


Test Results: {'eval_loss': 1.1808884143829346, 'eval_accuracy': 0.7752442996742671, 'eval_precision': 0.780187292494256, 'eval_recall': 0.7752442996742671, 'eval_f1': 0.775707664753159, 'eval_balanced_accuracy': 0.7168652239147367, 'eval_runtime': 2.5335, 'eval_samples_per_second': 363.534, 'eval_steps_per_second': 91.18, 'epoch': 10.0}
🧹 GPU memory cleared


In [19]:
trainer_1.save_model(trained_model_path_1)
base_tokenizer_1.save_pretrained(trained_tokenizer_path_1)

trainer_2.save_model(trained_model_path_2)
base_tokenizer_2.save_pretrained(trained_tokenizer_path_2)

('tokenizers/xlm-roberta-sentiment-shp/tokenizer_config.json',
 'tokenizers/xlm-roberta-sentiment-shp/special_tokens_map.json',
 'tokenizers/xlm-roberta-sentiment-shp/sentencepiece.bpe.model',
 'tokenizers/xlm-roberta-sentiment-shp/added_tokens.json',
 'tokenizers/xlm-roberta-sentiment-shp/tokenizer.json')